# Can we predict failure within 90 days of an inspection?
Target: `failed_90d`. Approach: audit the data first (leakage, censoring, drift), then compare a baseline against gradient boosting using a **time-ordered** split.

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, average_precision_score
from sklearn.inspection import permutation_importance
df = pd.read_csv('data.csv', parse_dates=['inspected_at']).sort_values('inspected_at').reset_index(drop=True)
print(df.shape, 'nulls:', int(df.isna().sum().sum()), 'unique assets:', df.asset_tag.nunique())
print('base rate:', df.failed_90d.mean().round(4), '| date range:', df.inspected_at.min(), '->', df.inspected_at.max())
df.describe().T

## 1. Data audit
### 1a. `parts_replaced` is leakage
It comes from the work order that *closed* the inspection record, so it may be written after the outcome is known.

In [ ]:
df.groupby('parts_replaced').failed_90d.agg(['mean','size'])

Every record with 2+ parts replaced failed, and none with 0 did. That looks like a repair following a failure, not something known at inspection time. **Excluded from the model**: using it would look near-perfect and be useless for real prediction. (If the work order is in fact closed at inspection time, before any outcome, this could be revisited; worth confirming with the data owner.)

### 1b. Failure rate over time (censoring and drift)

In [ ]:
q = df.groupby(df.inspected_at.dt.to_period('Q')).failed_90d.agg(['mean','size'])
display(q)
q['mean'].plot(kind='bar', title='Failure rate by inspection quarter'); plt.ylabel('rate'); plt.show()

The rate swings from ~3% to ~27% by quarter. The latest inspections (Q4 2025) are within 90 days of the end of the data, so some failures may not be observed yet (right-censoring), biasing those labels downward. The Q2-Q3 2024 spike is not explained by the available columns, so the base rate is not stable. A random split would overstate performance; we validate forward in time.

In [ ]:
feats = ['vibration_rms','bearing_temp_c','ambient_temp_c','meter_hours','load_pct']
print(df.groupby('site').failed_90d.mean().round(3).to_dict())
df.groupby('failed_90d')[feats].mean().T

In [ ]:
fig, ax = plt.subplots(1, 5, figsize=(18, 3))
for a, f in zip(ax, feats):
    for y_ in (0, 1):
        a.hist(df.loc[df.failed_90d == y_, f], bins=30, alpha=.6, density=True, label=str(y_))
    a.set_title(f)
ax[0].legend(title='failed'); plt.tight_layout(); plt.show()

## 2. Features and split
Each asset appears once, so there is no asset-level leakage across rows. Features: the five numeric measurements, `temp_rise = bearing_temp_c - ambient_temp_c`, and site (one-hot). No date features, since calendar effects would not generalise forward. Train on the earliest 70% of inspections, test on the latest 30%.

In [ ]:
X = df[feats].copy()
X['temp_rise'] = X.bearing_temp_c - X.ambient_temp_c
X = X.join(pd.get_dummies(df.site, prefix='site').astype(float))
y = df.failed_90d
cut = int(len(df) * .7)
Xtr, Xte, ytr, yte = X[:cut], X[cut:], y[:cut], y[cut:]
print('train', df.inspected_at[:cut].min().date(), '->', df.inspected_at[cut-1].date(), 'rate', ytr.mean().round(3))
print('test ', df.inspected_at[cut].date(), '->', df.inspected_at.max().date(), 'rate', yte.mean().round(3))

## 3. Models

In [ ]:
models = {
    'logistic': make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)),
    'hist_gb': HistGradientBoostingClassifier(max_depth=3, learning_rate=0.05, max_iter=200, random_state=0),
}
rows = []; preds = {}
for n, m in models.items():
    m.fit(Xtr, ytr); p = m.predict_proba(Xte)[:, 1]; preds[n] = p
    rows.append({'model': n, 'ROC AUC': roc_auc_score(yte, p), 'PR AUC': average_precision_score(yte, p)})
rows.append({'model': 'no-skill', 'ROC AUC': .5, 'PR AUC': yte.mean()})
pd.DataFrame(rows).round(3)

Bootstrap interval on held-out ROC AUC:

In [ ]:
rng = np.random.default_rng(0); yt = yte.to_numpy()
for n, p in preds.items():
    b = []
    for _ in range(500):
        i = rng.integers(0, len(yt), len(yt))
        if yt[i].min() != yt[i].max(): b.append(roc_auc_score(yt[i], p[i]))
    print(n, 'AUC 95% CI', np.percentile(b, [2.5, 97.5]).round(3))

### Feature importance (permutation, held-out)

In [ ]:
m = models['hist_gb']
r = permutation_importance(m, Xte, yte, scoring='roc_auc', n_repeats=10, random_state=0)
pd.Series(r.importances_mean, index=X.columns).sort_values().plot(kind='barh', title='Drop in test AUC when shuffled'); plt.show()

In [ ]:
p = preds['hist_gb']; t = pd.DataFrame({'p': p, 'y': yte.values})
t['decile'] = pd.qcut(t.p.rank(method='first'), 10, labels=False) + 1
display(t.groupby('decile').y.agg(['mean', 'size']).rename(columns={'mean': 'failure rate'}).round(3))
top = t.nlargest(int(len(t) * .1), 'p')
print('Top 10%% flagged: precision %.2f, captures %.0f%% of failures (base rate %.2f)' % (top.y.mean(), 100 * top.y.sum() / t.y.sum(), t.y.mean()))

## 4. Conclusion
- **Yes, but weakly.** On the forward-in-time test set, gradient boosting reaches ROC AUC ~0.75 (95% CI ~0.70-0.80) vs 0.68 for logistic regression. PR AUC is ~0.15 against a 0.056 base rate: about 2.6x better than chance, still low in absolute terms.
- Flagging the riskiest 10% of inspections catches ~27% of failures at ~15% precision. The model is useful for prioritising, not for confident individual calls.
- Failures are associated with higher vibration, bearing temperature and load, and with fewer meter hours; site has essentially no effect.
- **`parts_replaced` was deliberately excluded.** It separates the classes almost perfectly but is probably recorded after the fact. Confirm when it is written before ever using it.
- **Caveats:** the failure rate drifts a lot over time (14.5% in train, 5.6% in test), so absolute probabilities are not calibrated forward; recent labels may be censored; the test set has only ~100 failures. Next steps: confirm label completeness for late 2025, find what drove the 2024 Q2-Q3 spike, and add per-asset history if repeat inspections exist (here each asset has exactly one row).

*Note: the numbers above come from the run at the time of writing; re-run the notebook to refresh the outputs.*